# Porting a PyTorch LSTM to TPU with TorchTPU: a hands-on benchmark

*SoilTwin team, TorchTPU sprint, November 2026*

<a href="https://colab.research.google.com/github/areanddee/soiltwin-torchtpu-sprint-blog/blob/main/notebooks/soiltwin_torchtpu.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

We took an existing PyTorch model, an LSTM that predicts soil CO₂ flux from soil temperature, soil moisture and
weather, and trained it on an NVIDIA A100 and on Google TPU v5e and v6e using **TorchTPU**, which runs ordinary
PyTorch on TPUs as the device `tpu`.

This notebook has two parts:

1. **Run it yourself.** Train and test the model on whatever accelerator this runtime has, at two optimization levels,
   and see how your chip compares with ours. About 5 minutes on an accelerator.
2. **Our results on three chips.** The full benchmark: an optimization ladder, batch scaling, time to a usable model,
   and whether the chips produce the same model.

**Pick a runtime first.** In Colab: *Runtime → Change runtime type* → **v5e-1 TPU** or **v6e-1 TPU** (or a GPU such as
A100). On a CPU-only runtime everything still runs, with training cut to a few epochs.

## 1. Setup and the TorchTPU port

The repository is small: `soiltwin/model.py` (the network), `soiltwin/data.py` (loading and normalization),
`soiltwin/train.py` (the training loop, device handling and the optimization levels), and `data/soil_co2_traces.npz`.
It needs only PyTorch and NumPy; Colab runtimes already have both, and Colab's TPU runtimes come with TorchTPU.

In [ ]:
import os, sys
if not os.path.isdir('soiltwin'):
    if os.path.isdir('../soiltwin'):
        os.chdir('..')
    else:
        !git clone -q https://github.com/areanddee/soiltwin-torchtpu-sprint-blog.git /content/soiltwin-torchtpu-sprint-blog
        os.chdir('/content/soiltwin-torchtpu-sprint-blog')
sys.path.insert(0, os.getcwd()); sys.path.insert(0, os.path.join(os.getcwd(), 'figures'))

import inspect, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from soiltwin import train as T
from soiltwin.data import load_traces, CHANNELS
from soiltwin.model import CO2FluxLSTM
%matplotlib inline
plt.rcParams['figure.dpi'] = 110

device = T.pick_device('auto')
print('torch', torch.__version__)
print('device:', device, '|', T.device_name(device))

This is the entire port. TorchTPU registers the `tpu` device when `torch_tpu` is imported, and exposes it
through PyTorch's generic `torch.accelerator` API. The training code needs three things from us: pick the device, wait
for the device before reading a clock, and copy tensors to the CPU before saving a checkpoint so it loads anywhere.
Everything else, from `model.to(device)` to `loss.backward()` and `optimizer.step()`, is unchanged PyTorch.

In [ ]:
for f in (T.tpu_available, T.pick_device, T.sync):
    print(inspect.getsource(f))

## 2. The data

300 traces, each 6 weeks of half-hourly steps (2016) at one of 20 sites in 2023–2025. Inputs are soil temperature,
soil moisture, solar radiation (ERA5 SSRD), air temperature, day of year, and the initial microbial biomass. The target
is heterotrophic CO₂ flux from a soil-carbon process model driven by those inputs: the network learns to emulate it.
The split is random and fixed (seed 42): 240 training, 30 validation and 30 test traces.

In [ ]:
data = load_traces()
inputs, targets, meta = data
print(len(inputs), 'traces | inputs', inputs[0].shape, '| targets', targets[0].shape)
print('model channels:', CHANNELS)

i = 0
t = np.arange(2016) / 48
fig, axes = plt.subplots(4, 1, figsize=(9, 6), sharex=True)
for ax, (k, lab) in zip(axes[:3], [(0, 'soil temp (°C)'), (1, 'soil moisture'), (3, 'air temp (°C)')]):
    ax.plot(t, inputs[i][:, k], lw=1); ax.set_ylabel(lab)
axes[3].plot(t, targets[i][:, 2], lw=1.2, color='k'); axes[3].set_ylabel('CO₂ flux\n(µmol m⁻² s⁻¹)')
axes[3].set_xlabel('days'); axes[0].set_title(f"trace {i}: site {meta[i]['site_id']}, {meta[i]['year']}", loc='left')
plt.tight_layout()

## 3. The model

A 2-layer LSTM (hidden size 64) reads the 2016 steps in order. A temporal-attention gate and a small bidirectional
LSTM that detects the onset of microbial colonization reweight its output, and a per-trace scale-and-shift head sets
the amplitude. It is a small model, about 70 thousand parameters, and that turns out to matter for the hardware.

In [ ]:
print(CO2FluxLSTM(input_size=6))
print(f'{CO2FluxLSTM(input_size=6).count_parameters():,} trainable parameters')

## 4. Train it: the naive port

Level 0 is the training loop exactly as it ran on GPUs before the port: a `DataLoader` copies each batch from the host,
and the loop calls `.item()` on the losses every batch. Watch the first epoch: on a TPU it includes compiling the
training step, which happens once per job.

In [ ]:
EPOCHS = 100 if device.type != 'cpu' else 6     # early stopping usually ends an accelerator run at 55-100 epochs
yours = []
out = T.train(data, opt_level=0, batch_size=32, epochs=EPOCHS, device=device)
yours.append(out)
r = out['results']
print(f"\n{r['device_name']}: first epoch {r['first_epoch_s']:.2f} s, then {r['steady_epoch_s']:.3f} s/epoch; "
      f"{r['epochs_run']} epochs in {r['train_wall_s']:.0f} s; test R² {r['test_r2']:.3f}")

## 5. Train it again: device-resident data, bigger batch

Level 1 keeps the running losses and validation metrics on the device and reads them back once per epoch instead of
three times per batch. Level 2 also uploads the whole dataset to the device once and shuffles it there, so there are no
per-batch host-to-device copies. Here we run level 2 with batch 128, i.e. 2 optimizer steps per epoch instead of 8.

This run's first epoch is shorter than in our reference runs (about 6 s instead of 11 s on a v5e): it shares the
Python process with the level-0 run above, so some compiled work is reused. Our reference runs each started fresh.

```python
# level 0: three host syncs per batch            # level 1: one per epoch
total_loss += loss.item()                         totals += torch.stack([loss.detach(), mse.detach(), colon.detach()])
                                                  ...
                                                  loss, mse, colon = (totals / n_batches).tolist()
```

In [ ]:
out = T.train(data, opt_level=2, batch_size=128, epochs=EPOCHS, device=device)
yours.append(out)
r = out['results']
print(f"\n{r['device_name']}: first epoch {r['first_epoch_s']:.2f} s, then {r['steady_epoch_s']:.3f} s/epoch; "
      f"{r['epochs_run']} epochs in {r['train_wall_s']:.0f} s; test R² {r['test_r2']:.3f}")

## 6. Inference on the test set

The best checkpoint from the naive run predicts all 30 held-out traces (60,480 half-hour steps) in one pass on the
device. Validation during training already compiled the forward pass, so even the first call here is fast.

In [ ]:
best = yours[0]
model, norm = best['model'], best['normalizer']
loaders, _, _ = T.make_loaders(data, opt_level=2, batch_size=30, device=device)
x, y, mask, _ = next(iter(loaders['test']))
model.eval()
with torch.no_grad():
    T.sync(device); t0 = time.perf_counter(); model(x, mask); T.sync(device); first = time.perf_counter() - t0
    reps = []
    for _ in range(5):
        t0 = time.perf_counter(); pred = model(x, mask); T.sync(device); reps.append(time.perf_counter() - t0)
print(f'inference, 30 traces x 2016 steps: first call {first * 1e3:.0f} ms, then {np.median(reps) * 1e3:.1f} ms')

te = best['test']
m = te['mask'] > 0
obs, prd = te['targets'][m], te['preds'][m]
fig, (a, b) = plt.subplots(1, 2, figsize=(11, 3.8), gridspec_kw={'width_ratios': [1, 2]})
a.hexbin(obs, prd, gridsize=50, bins='log', cmap='Greys', mincnt=1)
a.plot([0, 3.5], [0, 3.5], 'r--', lw=1); a.set_xlabel('observed flux'); a.set_ylabel('predicted flux')
a.set_title(f"all test steps, R² {best['results']['test_r2']:.3f}", loc='left')
k = int(np.argmax(te['targets'].std(axis=1)))
b.plot(np.arange(2016) / 48, te['targets'][k], 'k', lw=1.2, label='observed')
b.plot(np.arange(2016) / 48, te['preds'][k], color='#7a4fc9', lw=1.2, label='predicted')
b.set_xlabel('days'); b.set_ylabel('CO₂ flux (µmol m⁻² s⁻¹)'); b.legend()
b.set_title(f"most variable test trace: site {te['site_id'][k]}, {te['year'][k]}", loc='left')
plt.tight_layout()

## 7. Your chip next to ours

Your two runs (violet) against the same configurations on our A100, TPU v5e and TPU v6e. On a CPU runtime the run
stopped after a few epochs, so the learning curve ends early and the test R² is low.

In [ ]:
import your_chip
your_chip.compare(yours);

---
# Part 2: our results on three chips

Same model, same data, same command on each chip: Colab's NVIDIA A100-SXM4-40GB (PyTorch 2.11, CUDA 13.0) and Colab's
TPU v5e and TPU v6e (PyTorch 2.14.1 with TorchTPU; identical software on both TPUs). Six configurations per chip,
`oL_bB` = optimization level L, batch size B. The figures below read `results/runs.json` and `results/preds/`.

In [ ]:
import json, pandas as pd, _accel_style
_accel_style.save = lambda fig, name: None   # draw inline only
runs = json.load(open('results/runs.json'))['runs']
order = {c: i for i, c in enumerate(_accel_style.CHIP_ORDER)}
df = pd.DataFrame([{'chip': r['chip'], 'run': r['config'], 'test R²': r['test_r2'],
                    'best / run epochs': f"{r['best_epoch']} / {r['epochs_run']}",
                    'first epoch (s)': r['first_epoch_s'], 'steady epoch (s)': r['steady_epoch_s'],
                    'traces / s': r['throughput_traces_per_s'], 'time to R² 0.90 (s)': r['time_to_target_s'],
                    'job wall clock (s)': r['real_s']} for r in runs])
df['_c'] = df['chip'].map(order); df['_r'] = df['run'].map(lambda s: (int(s[1]), int(s.split('_b')[1])))
df.sort_values(['_c', '_r']).drop(columns=['_c', '_r']).set_index(['chip', 'run']).round(
    {'test R²': 3, 'first epoch (s)': 2, 'steady epoch (s)': 3, 'traces / s': 0, 'time to R² 0.90 (s)': 1, 'job wall clock (s)': 0})

### Fixed costs dominate a one-minute job

On the TPUs the first epoch of the naive port takes 17–19 s because it compiles the training step; after that every
epoch reuses the compiled program (epoch times are flat, so nothing recompiles). On the A100 the first epoch costs 2.6 s,
but start-up, data loading and test evaluation take 25 s against the TPUs' 8 s.

In [ ]:
import fig10_time_breakdown; fig10_time_breakdown.main()

### GPU-style optimizations help the GPU, not the TPU

Removing per-batch host syncs and keeping the data on the device give the A100 1.26× and 1.39×. Both TPUs ignore them,
within 2%. Our reading: a TPU epoch here is dominated by executing the compiled step itself, so the host round-trips that
stall a GPU are a negligible share of it.

In [ ]:
import fig8_accel_ladder; fig8_accel_ladder.main()

### On a TPU the lever is the batch size, but bigger batches are not free

Larger batches mean fewer, bigger steps per epoch, and throughput rises on every chip, most on the v6e. But at a fixed
learning rate the model needs roughly the same number of *optimizer steps* to become usable whatever the batch, so a
bigger batch mostly means more epochs. Batches of 128 and 240 also end slightly less accurate within 100 epochs.

In [ ]:
import fig9_batch_scaling; fig9_batch_scaling.main()

### Time to a usable model

Training time until validation R² first reaches 0.90 (dashed: naive port; solid: the fastest configuration per chip).

In [ ]:
import fig11_time_to_accuracy; fig11_time_to_accuracy.main()

### TPU v6e vs TPU v5e

We expected the newer chip to be faster. For this model it mostly isn't:

In [ ]:
rows = []
for lvl, bs in [(0, 32), (1, 32), (2, 32), (2, 64), (2, 128), (2, 240)]:
    a, b = _accel_style.get(runs, 'TPU v5e', lvl, bs), _accel_style.get(runs, 'TPU v6e', lvl, bs)
    rows.append({'run': f'o{lvl}_b{bs}', 'optimizer steps / epoch': -(-240 // bs), 'v5e steady epoch (s)': a['steady_epoch_s'],
                 'v6e steady epoch (s)': b['steady_epoch_s'], 'v6e speedup over v5e': a['steady_epoch_s'] / b['steady_epoch_s']})
pd.DataFrame(rows).set_index('run').round(3)

At batch 32 (8 optimizer steps per epoch) a v6e epoch takes about 30% longer than a v5e epoch; only at batch 240
(one step per epoch) does it pull ahead, by about 10%. The v6e has far more peak compute, but this model cannot use it:
an LSTM walks 2016 time steps one after another with tiny (64-wide) matrix products, so each step is bound by latency
and per-step overhead rather than arithmetic. Our working hypothesis is that the v6e's fixed cost per step is higher on
this workload; we have not profiled it. A wider model or a far larger batch would be needed to show the v6e's advantage.

### Are the answers the same?

Every half-hour step of the 30 test traces from the naive port. Top: each chip against the observations. Bottom: chip
against chip. The two TPU generations produce almost the same model (r 0.9997): their training curves match to about
1e-4 early on and both stop at the same epoch. The A100 model differs from either TPU model by about as much as each misses
the observations: floating-point arithmetic differs between GPU and TPU, and training amplifies the difference. All runs
use the same seed, and repeat runs on the A100 and on the v5e reproduced exactly, so these are differences of numerics,
not of luck.

In [ ]:
import fig12_pred_vs_obs; fig12_pred_vs_obs.main()

Pooled R² is dominated by the high-flux traces. Per trace, all three models have skill wherever the flux actually
varies, and the error never drops below about 0.01 µmol m⁻² s⁻¹; on near-zero-flux traces that floor is larger than the
signal, so per-trace R² goes negative even though the absolute error is tiny.

In [ ]:
import fig14_per_trace_skill; fig14_per_trace_skill.main()

In [ ]:
import fig13_sample_traces; fig13_sample_traces.main()

## Takeaways

1. **Porting to TorchTPU was easy.** The same PyTorch training code ran on TPU v5e and v6e with a few lines of device
   handling, and produced models of the same quality as the A100.
2. **TPU compile time is a fixed cost**: 7–19 s in the first epoch of every job. After that, epochs are flat.
3. **GPU optimizations don't transfer.** Removing host syncs and keeping data on the device sped up the A100 by up to
   1.39× and did nothing for either TPU. Batch size is the TPU lever, but at a fixed learning rate it trades fewer steps
   per epoch for more epochs.
4. **For a small, sequential model the A100 wins**, by 4–5× per epoch and 8–10× in time to a usable model. TPUs are
   built for wide matrix products and large batches; a 64-wide LSTM offers neither.
5. **A newer TPU is not automatically faster.** The v6e was about 30% slower than the v5e at small batches, 10% faster
   at the largest, and trained essentially the same model.

## Run it from the command line

```bash
python -m soiltwin.train --opt-level 0 --batch-size 32     # naive port
python -m soiltwin.train --opt-level 2 --batch-size 128    # device-resident data, bigger batch
```

## Data and licence

Code: Apache-2.0. The training data combine synthetic site data with ERA5 reanalysis. *Contains modified Copernicus
Climate Change Service information (2026).* Hersbach, H. et al. (2020), The ERA5 global reanalysis,
*Q. J. R. Meteorol. Soc.* 146, 1999–2049. See `data/README.md`.